### SET UP

In [ ]:
import os
import sys
import gc
import json
import time
import random
from datetime import datetime
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np

# Nạp các module tự viết từ thư mục lõi
sys.path.append(os.path.abspath("../"))
from core.module05_fusion_classifier.dataset import VariantFusionDataset
from core.module05_fusion_classifier.fusion_model import MultiStrategyFusionModel
from core.module05_fusion_classifier.xgboost_model import XGBoostFusionManager
from core.module05_fusion_classifier.evaluator_profiler import FusionEvaluatorProfiler

# ==============================================================================
# 0. HẠT GIỐNG TÁI LẬP (REPRODUCIBILITY SEED)
# ==============================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"[*] Đã khóa Seed = {seed} cho toàn bộ hệ thống.")

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang sử dụng thiết bị: {DEVICE}")

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN 
# ==============================================================================
REPO_ROOT = os.path.abspath("../")
BIO_DIR = f"{REPO_ROOT}/data/processed"          
GEOM_DIR = f"{REPO_ROOT}/data/processed"         
EMBED_DIR = f"{REPO_ROOT}/data/processed"        
FM_JSON_PATH = f"{REPO_ROOT}/data/processed/fm_profiling.json"

current_time = datetime.now().strftime("%Y%m%d_%H%M")
BATCH_RUN_DIR = f"{REPO_ROOT}/experiments/batch_run_{current_time}"
os.makedirs(BATCH_RUN_DIR, exist_ok=True)
print(f"[*] Thư mục lưu trữ thực nghiệm: {BATCH_RUN_DIR}")

# ==============================================================================
# 2. KHÔNG GIAN DỮ LIỆU VÀ MÔ HÌNH (CONFIG SPACE)
# ==============================================================================
CONFIG = {
    "batch_size": 256,
    "epochs": 15,
    "lr": 1e-4,
    "num_workers": 0
}

# [TÍNH NĂNG MỚI] Hỗ trợ duyệt qua nhiều bộ Datasets khác nhau
DATASETS = [
    # Fold 1 đánh giá trên ClinVar
    {"name": "Fold1_ClinVar", "train": "train_fold1", "val": "val_fold1", "test": "test_clinvar_hq"},
    # Fold 1 đánh giá trên gnomAD
    {"name": "Fold1_gnomAD", "train": "train_fold1", "val": "val_fold1", "test": "test_gnomad"},
    
    # Fold 2 đánh giá trên ClinVar
    {"name": "Fold2_ClinVar", "train": "train_fold2", "val": "val_fold2", "test": "test_clinvar_hq"},
    # Fold 2 đánh giá trên gnomAD
    {"name": "Fold2_gnomAD", "train": "train_fold2", "val": "val_fold2", "test": "test_gnomad"},
    
    #... (Có thể thêm Fold 3, 4, 5)
]

MODELS_SPACE = [
    {"name": "nt_v1_500m", "seq_type": "dna"},
    {"name": "nt_v3_650m", "seq_type": "dna"},
    {"name": "evo2_1b", "seq_type": "dna"},
    {"name": "esm1b_650m", "seq_type": "protein"},
    {"name": "esm2_650m", "seq_type": "protein"},
    {"name": "esmc_600m", "seq_type": "protein"}
]

DNA_MODELS = [m for m in MODELS_SPACE if m['seq_type'] == 'dna']
PROT_MODELS = [m for m in MODELS_SPACE if m['seq_type'] == 'protein']

POOLING_STRATEGIES = ["center", "cls", "mean"]

EXPERIMENTS = [
    {"name": "PyTorch_Concat", "type": "pytorch", "fusion": "concat"},
    {"name": "PyTorch_CrossAttn", "type": "pytorch", "fusion": "cross_attention"},
    {"name": "PyTorch_Transformer", "type": "pytorch", "fusion": "transformer"},
    {"name": "PyTorch_Gating", "type": "pytorch", "fusion": "gating"},
    {"name": "Pure_XGBoost_Concat", "type": "xgboost_pure", "fusion": "concat"},
    {"name": "Hybrid_Concat_XGBoost", "type": "hybrid", "fusion": "concat"}, 
    {"name": "Hybrid_CrossAttn_XGBoost", "type": "hybrid", "fusion": "cross_attention"},
    {"name": "Hybrid_Transformer_XGBoost", "type": "hybrid", "fusion": "transformer"},
    {"name": "Hybrid_Gating_XGBoost", "type": "hybrid", "fusion": "gating"}
]

# ==============================================================================
# 3. CÁC HÀM TIỆN ÍCH LÕI (CORE HELPERS)
# ==============================================================================

# [TÍNH NĂNG MỚI] Auto-Dim: Đọc chớp nhoáng Dimension từ file .pt
DIM_CACHE = {}
def get_cached_dim(model_name, split_name, pooling_strategy):
    if model_name == "None": return 0
    key = f"{model_name}_{pooling_strategy}"
    if key not in DIM_CACHE:
        pt_path = f"{EMBED_DIR}/{split_name}/{model_name}_{pooling_strategy}.pt"
        if not os.path.exists(pt_path): raise FileNotFoundError(f"[LỖI AUTO-DIM] Không tìm thấy {pt_path}")
        
        print(f"    [*] Auto-Dim: Phân tích {model_name} ({pooling_strategy})...")
        data = torch.load(pt_path, weights_only=False)
        DIM_CACHE[key] = data["E_ref"][0].shape[0]
        del data
    return DIM_CACHE[key]

def get_dataloaders(split_name, pooling_strategy, dna_name, prot_name, active_mods, is_train=True):
    dataset = VariantFusionDataset(
        bio_parquet_path=f"{BIO_DIR}/{split_name}_normalized.parquet",
        dna_geom_path=f"{GEOM_DIR}/{split_name}/{dna_name}_geom_norm.parquet" if dna_name != "None" else None,
        prot_geom_path=f"{GEOM_DIR}/{split_name}/{prot_name}_geom_norm.parquet" if prot_name != "None" else None,
        dna_pt_path=f"{EMBED_DIR}/{split_name}/{dna_name}_{pooling_strategy}.pt" if dna_name != "None" else None,
        prot_pt_path=f"{EMBED_DIR}/{split_name}/{prot_name}_{pooling_strategy}.pt" if prot_name != "None" else None,
        active_modalities=active_mods, is_train=True 
    )
    return DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=is_train, num_workers=CONFIG["num_workers"], pin_memory=True)

def safe_tensor(batch, key): return batch[key].to(DEVICE) if key in batch else None
def safe_slice_dummy(batch, key):
    t = safe_tensor(batch, key)
    return t[:1] if t is not None else None

def extract_features_for_ml(model, dataloader, extract_f_global=True):
    model.eval()
    all_f_global, all_v_dna, all_v_prot, all_bg, all_labels, all_vids = [], [], [], [], [], []
    with torch.no_grad():
        for batch in dataloader:
            v_dna = safe_tensor(batch, "v_dna")
            v_prot = safe_tensor(batch, "v_prot")
            
            bg_list = []
            if "bio_features" in batch: bg_list.append(batch["bio_features"])
            if "geom_features" in batch: bg_list.append(batch["geom_features"])
            bg = torch.cat(bg_list, dim=-1).to(DEVICE) if len(bg_list) > 0 else None
            
            if extract_f_global:
                f_global = model(v_dna, v_prot, safe_tensor(batch, "bio_features"), safe_tensor(batch, "geom_features"), return_features=True)
                all_f_global.append(f_global.cpu())
                
            if v_dna is not None: all_v_dna.append(v_dna.cpu())
            if v_prot is not None: all_v_prot.append(v_prot.cpu())
            if bg is not None: all_bg.append(bg.cpu())
            
            all_labels.append(batch["label"].cpu())
            all_vids.extend(batch["variant_id"])
            
    return (
        torch.cat(all_f_global).numpy() if extract_f_global else None,
        torch.cat(all_v_dna).numpy() if len(all_v_dna) > 0 else None,
        torch.cat(all_v_prot).numpy() if len(all_v_prot) > 0 else None,
        torch.cat(all_bg).numpy() if len(all_bg) > 0 else None,
        torch.cat(all_labels).squeeze(-1).numpy(), all_vids
    )

# ==============================================================================
# HÀM CHẠY THEO NHÓM (TRÁNH LOAD DATA NHIỀU LẦN)
# ==============================================================================
def execute_ablation_group(dna_cfg, prot_cfg, pooling, active_mods, dataset_cfg, completed_keys, g_metrics, g_profiling):
    dna_name = dna_cfg["name"] if (dna_cfg and 'dna' in active_mods) else "None"
    prot_name = prot_cfg["name"] if (prot_cfg and 'prot' in active_mods) else "None"
    ablation_str = "_".join(sorted(active_mods))
    num_seq = int('dna' in active_mods) + int('prot' in active_mods)

    # 1. Kiểm duyệt xem nhóm này đã chạy xong chưa (Tránh nạp Dataloader dư thừa)
    tasks_to_run = []
    for exp in EXPERIMENTS:
        if num_seq < 2 and exp["fusion"] in ["cross_attention", "transformer", "gating"]: continue
        run_key = f"{dataset_cfg['name']}_{pooling}_{dna_name}_{prot_name}_{ablation_str}_{exp['name']}"
        if run_key not in completed_keys: tasks_to_run.append((exp, run_key))
        else: print(f"    -> [CACHE HIT] Bỏ qua {run_key}")

    if not tasks_to_run: return []

    # 2. Lấy Kích thước chiều tự động
    dna_dim = get_cached_dim(dna_name, dataset_cfg['train'], pooling)
    prot_dim = get_cached_dim(prot_name, dataset_cfg['train'], pooling)

    # 3. Nạp Data
    print(f"\n  [>] Tải Dữ liệu cho Nhóm: DNA={dna_name}, PROT={prot_name}, ABLATION={ablation_str}")
    train_loader = get_dataloaders(dataset_cfg['train'], pooling, dna_name, prot_name, active_mods, True)
    val_loader = get_dataloaders(dataset_cfg['val'], pooling, dna_name, prot_name, active_mods, False)
    test_loader = get_dataloaders(dataset_cfg['test'], pooling, dna_name, prot_name, active_mods, False)

    group_results = []
    for exp, run_key in tasks_to_run:
        completed_keys.add(run_key)
        exp_name = f"{dataset_cfg['name']}_{pooling}_{ablation_str}_{dna_name}_{prot_name}_{exp['name']}"
        print(f"      - Đang huấn luyện: {exp['name']}")
        
        exp_dir = f"{BATCH_RUN_DIR}/{exp_name}"
        os.makedirs(f"{exp_dir}/checkpoints", exist_ok=True)
        best_model_path = f"{exp_dir}/checkpoints/best_model.pth"
        
        profiler = FusionEvaluatorProfiler(f"{exp_dir}/tensorboard_logs", FM_JSON_PATH, DEVICE)
        model = MultiStrategyFusionModel(dna_in_dim=dna_dim, prot_in_dim=prot_dim, active_modalities=active_mods, fusion_strategy=exp["fusion"]).to(DEVICE)
        test_metrics, e2e_profiling, vids_t, y_true_t, y_probs_t, y_preds_t = {}, {}, [], [], [], []
        
        dummy_batch = next(iter(val_loader))
        d_in_safe = (safe_slice_dummy(dummy_batch, "v_dna"), safe_slice_dummy(dummy_batch, "v_prot"), 
                     safe_slice_dummy(dummy_batch, "bio_features"), safe_slice_dummy(dummy_batch, "geom_features"))

        # --- PYTORCH E2E ---
        if exp["type"] == "pytorch":
            profiler.profile_pytorch_fusion(model, d_in_safe)
            optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
            criterion = nn.BCEWithLogitsLoss()
            best_val_mcc = -1.0
            
            for epoch in range(CONFIG["epochs"]):
                model.train()
                for batch in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(safe_tensor(batch, "v_dna"), safe_tensor(batch, "v_prot"), safe_tensor(batch, "bio_features"), safe_tensor(batch, "geom_features")), batch["label"].to(DEVICE))
                    loss.backward()
                    optimizer.step()
                    
                model.eval()
                all_probs_v, all_preds_v, all_labels_v = [], [], []
                with torch.no_grad():
                    for batch in val_loader:
                        logits = model(safe_tensor(batch, "v_dna"), safe_tensor(batch, "v_prot"), safe_tensor(batch, "bio_features"), safe_tensor(batch, "geom_features"))
                        probs = torch.sigmoid(logits)
                        all_probs_v.append(probs.cpu())
                        all_preds_v.append((probs > 0.5).float().cpu())
                        all_labels_v.append(batch["label"].cpu())
                        
                val_metrics = profiler.compute_metrics(torch.cat(all_labels_v).squeeze(-1).numpy(), torch.cat(all_probs_v).squeeze(-1).numpy(), torch.cat(all_preds_v).squeeze(-1).numpy())
                if val_metrics["MCC"] > best_val_mcc:
                    best_val_mcc = val_metrics["MCC"]
                    torch.save(model.state_dict(), best_model_path)
            
            model.load_state_dict(torch.load(best_model_path, weights_only=True))
            model.eval()
            profiler.reset_memory_stats()
            all_probs_t, all_preds_t, all_labels_t = [], [], []
            with torch.no_grad():
                for batch in test_loader:
                    profiler.tic_inference()
                    logits = model(safe_tensor(batch, "v_dna"), safe_tensor(batch, "v_prot"), safe_tensor(batch, "bio_features"), safe_tensor(batch, "geom_features"))
                    profiler.toc_inference()
                    probs = torch.sigmoid(logits)
                    all_probs_t.append(probs.cpu())
                    all_preds_t.append((probs > 0.5).float().cpu())
                    all_labels_t.append(batch["label"].cpu())
                    vids_t.extend(batch["variant_id"])
                    
            profiler.finalize_fusion_inference_profiling(len(test_loader.dataset))
            y_probs_t, y_preds_t, y_true_t = torch.cat(all_probs_t).squeeze(-1).numpy(), torch.cat(all_preds_t).squeeze(-1).numpy(), torch.cat(all_labels_t).squeeze(-1).numpy()
            test_metrics = profiler.compute_metrics(y_true_t, y_probs_t, y_preds_t)
            e2e_profiling = profiler.get_e2e_profiling(dna_name, prot_name, active_mods)

        # --- PURE XGBOOST ---
        elif exp["type"] == "xgboost_pure":
            _, dna_tr, prot_tr, bg_tr, y_tr, _ = extract_features_for_ml(model, train_loader, False)
            _, dna_vl, prot_vl, bg_vl, y_vl, _ = extract_features_for_ml(model, val_loader, False)
            _, dna_ts, prot_ts, bg_ts, y_ts, vids_t = extract_features_for_ml(model, test_loader, False)
            
            xgb_manager = XGBoostFusionManager(pca_components=256)
            xgb_manager.train_pure(dna_tr, prot_tr, bg_tr, y_tr, dna_vl, prot_vl, bg_vl, y_vl)
            
            profiler.reset_memory_stats()
            profiler.tic_inference()
            y_probs_t, y_preds_t, _ = xgb_manager.predict_pure(dna_ts, prot_ts, bg_ts)
            profiler.toc_inference()
            
            profiler.finalize_fusion_inference_profiling(len(test_loader.dataset))
            y_true_t = y_ts
            test_metrics = profiler.compute_metrics(y_true_t, y_probs_t, y_preds_t)
            e2e_profiling = profiler.get_e2e_profiling(dna_name, prot_name, active_mods)
            del xgb_manager, dna_tr, prot_tr, bg_tr, dna_vl, prot_vl, bg_vl, dna_ts, prot_ts, bg_ts

        # --- HYBRID XGBOOST ---
        elif exp["type"] == "hybrid":
            profiler.profile_pytorch_fusion(model, d_in_safe)
            optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
            criterion = nn.BCEWithLogitsLoss()
            
            model.train()
            for _ in range(2): # Lướt nhanh 2 Epoch
                for batch in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(safe_tensor(batch, "v_dna"), safe_tensor(batch, "v_prot"), safe_tensor(batch, "bio_features"), safe_tensor(batch, "geom_features")), batch["label"].to(DEVICE))
                    loss.backward()
                    optimizer.step()
                    
            f_glob_tr, _, _, _, y_tr, _ = extract_features_for_ml(model, train_loader, True)
            f_glob_vl, _, _, _, y_vl, _ = extract_features_for_ml(model, val_loader, True)
            f_glob_ts, _, _, _, y_ts, vids_t = extract_features_for_ml(model, test_loader, True)
            
            xgb_manager = XGBoostFusionManager()
            xgb_manager.train_hybrid(f_glob_tr, y_tr, f_glob_vl, y_vl)
            
            profiler.reset_memory_stats()
            profiler.tic_inference()
            y_probs_t, y_preds_t, _ = xgb_manager.predict_hybrid(f_glob_ts)
            profiler.toc_inference()
            
            profiler.finalize_fusion_inference_profiling(len(test_loader.dataset))
            y_true_t = y_ts
            test_metrics = profiler.compute_metrics(y_true_t, y_probs_t, y_preds_t)
            e2e_profiling = profiler.get_e2e_profiling(dna_name, prot_name, active_mods)
            del xgb_manager, f_glob_tr, f_glob_vl, f_glob_ts

        # --- GHI KẾT QUẢ ---
        pd.DataFrame({"Variant_ID": vids_t, "True_Label": y_true_t, "Predicted_Probability": y_probs_t, "Prediction_Class": y_preds_t}).to_csv(f"{exp_dir}/test_probabilities.csv", index=False)
        profiler.log_hparams(hparam_dict={"fusion": exp["fusion"], "pooling": pooling, "ablation": ablation_str}, final_metrics={"MCC_Test": test_metrics.get("MCC", 0)})
        profiler.close()
        
        row_metrics = {"Dataset": dataset_cfg['name'], "Pooling": pooling, "Ablation": ablation_str, "DNA_Model": dna_name, "Prot_Model": prot_name, "Network": exp["name"], **test_metrics}
        g_metrics.append(row_metrics)
        g_profiling.append({"Experiment": exp_name, **e2e_profiling})
        group_results.append(row_metrics)
        
        del model, profiler
        torch.cuda.empty_cache()
        gc.collect()
        
    del train_loader, val_loader, test_loader
    gc.collect()
    return group_results

### The Master Loop

In [ ]:
# ==============================================================================
# 4. CHU TRÌNH TỰ ĐỘNG: TÌM CẶP VÔ ĐỊCH TOÀN CỤC & ABLATION CẮT LỚP
# ==============================================================================
global_leaderboard_metrics = []
global_leaderboard_profiling = []

for pooling in POOLING_STRATEGIES:
    print("\n" + "="*90)
    print(f"🚀 KHỞI ĐỘNG CHIẾN LƯỢC POOLING: {pooling.upper()}")
    print("="*90)
    
    completed_keys = set() # Bộ nhớ đệm chống chạy trùng lắp

    # ======================================================================
    # GIAI ĐOẠN 1: TÌM CẶP TỐT NHẤT TRÊN *TẤT CẢ* TẬP DỮ LIỆU (GLOBAL SEARCH)
    # ======================================================================
    print("\n  [STAGE 1] Chạy vét cạn 9 tổ hợp DNA x Protein trên TOÀN BỘ Datasets...")
    for ds_cfg in DATASETS:
        print(f"    -> Đang đánh giá trên tập: {ds_cfg['name']}")
        for dna_m in DNA_MODELS:
            for prot_m in PROT_MODELS:
                execute_ablation_group(dna_m, prot_m, pooling, ["dna", "prot"], ds_cfg, completed_keys, global_leaderboard_metrics, global_leaderboard_profiling)

    # ----------------------------------------------------------------------
    # TÍNH TOÁN "MAX CAPACITY": MAX(Networks) -> MEAN(Datasets)
    # ----------------------------------------------------------------------
    # 1. Nhóm điểm theo (DNA, Prot, Dataset)
    pair_ds_scores = {}
    for m in global_leaderboard_metrics:
        if m["Pooling"] == pooling and m["Ablation"] == "dna_prot":
            key = (m["DNA_Model"], m["Prot_Model"], m["Dataset"])
            if key not in pair_ds_scores: pair_ds_scores[key] = []
            pair_ds_scores[key].append(m["MCC"])

    if pair_ds_scores:
        # 2. Tìm điểm MAX (Mạng tốt nhất) cho mỗi cặp trên mỗi Dataset
        pair_ds_max = {k: np.max(v) for k, v in pair_ds_scores.items()}
        
        # 3. Gom nhóm theo Cặp (DNA, Prot) để tính trung bình qua các Datasets
        pair_avg_max = {}
        for (dna, prot, ds), max_mcc in pair_ds_max.items():
            pair = (dna, prot)
            if pair not in pair_avg_max: pair_avg_max[pair] = []
            pair_avg_max[pair].append(max_mcc)
            
        # 4. Tính Mean MCC cuối cùng và xướng tên nhà vô địch
        final_scores = {k: np.mean(v) for k, v in pair_avg_max.items()}
        best_pair_key = max(final_scores, key=final_scores.get)
        
        best_dna = next(m for m in DNA_MODELS if m["name"] == best_pair_key[0])
        best_prot = next(m for m in PROT_MODELS if m["name"] == best_pair_key[1])
        
        print(f"\n  🏆 [KẾT QUẢ STAGE 1] TỔ HỢP VÔ ĐỊCH TOÀN CỤC (GLOBAL BEST PAIR) 🏆")
        print(f"      DNA: {best_dna['name']} | PROTEIN: {best_prot['name']}")
        print(f"      (Đánh giá dựa trên Mạng tốt nhất của mỗi tập: Mean MCC = {final_scores[best_pair_key]:.4f})")
    else:
        best_dna, best_prot = DNA_MODELS[0], PROT_MODELS[0]

    # ======================================================================
    # GIAI ĐOẠN 2, 3, 4: ABLATION CẮT LỚP (Chỉ dùng mô hình Vô địch)
    # ======================================================================
    print("\n  [STAGE 2, 3, 4] Khởi chạy Ablation với Cặp Vô Địch trên tất cả Datasets...")
    for ds_cfg in DATASETS:
        print(f"\n    [>>] Áp dụng Ablation trên tập: {ds_cfg['name']}")
        
        # STAGE 2: Đơn phương thức
        execute_ablation_group(best_dna, None, pooling, ["dna"], ds_cfg, completed_keys, global_leaderboard_metrics, global_leaderboard_profiling)
        execute_ablation_group(None, best_prot, pooling, ["prot"], ds_cfg, completed_keys, global_leaderboard_metrics, global_leaderboard_profiling)

        # STAGE 3: Mở rộng Ngữ cảnh
        execute_ablation_group(best_dna, best_prot, pooling, ["dna", "prot", "geom"], ds_cfg, completed_keys, global_leaderboard_metrics, global_leaderboard_profiling)
        execute_ablation_group(best_dna, best_prot, pooling, ["dna", "prot", "bio"], ds_cfg, completed_keys, global_leaderboard_metrics, global_leaderboard_profiling)
        execute_ablation_group(best_dna, best_prot, pooling, ["dna", "prot", "bio", "geom"], ds_cfg, completed_keys, global_leaderboard_metrics, global_leaderboard_profiling)

        # STAGE 4: ML Cổ điển
        execute_ablation_group(None, None, pooling, ["bio", "geom"], ds_cfg, completed_keys, global_leaderboard_metrics, global_leaderboard_profiling)
        
# ==============================================================================
# 5. XUẤT BẢNG XẾP HẠNG TỔNG HỢP (THE GLOBAL LEADERBOARDS)
# ==============================================================================
print("\n" + "="*80)
print("[LEADERBOARD 1] HIỆU SUẤT PHÂN LOẠI (CLASSIFICATION METRICS)")
print("="*80)
df_metrics = pd.DataFrame(global_leaderboard_metrics).sort_values(by=["Dataset", "MCC"], ascending=[True, False])
display(df_metrics)
df_metrics.to_csv(f"{BATCH_RUN_DIR}/global_leaderboard_metrics.csv", index=False)

print("\n" + "="*80)
print("[LEADERBOARD 2] HIỆU NĂNG TÀI NGUYÊN (END-TO-END PROFILING)")
print("="*80)
df_profiling = pd.DataFrame(global_leaderboard_profiling)
display(df_profiling)
df_profiling.to_csv(f"{BATCH_RUN_DIR}/global_leaderboard_profiling.csv", index=False)

print(f"\n[THÀNH CÔNG] Master Pipeline đã cày xong! Báo cáo lưu tại: {BATCH_RUN_DIR}")